In [1]:
print("hello")

hello


In [20]:
from openai import OpenAI

from dotenv import load_dotenv
import os
import gradio as gr
import json

load_dotenv()

Openai_api_key = os.getenv("OPENAI_API_KEY")




In [3]:
import sqlite3

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY,price REAL)")
    conn.commit()


In [30]:
def get_price_tool(city):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE CITY = ?",(city.lower(),))
        result = cursor.fetchone()
        return f"The ticket price for {city} is {result[0]}" if result else "There is no data for this request"


In [5]:
def insert_db_records_tool(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO prices (city, price) VALUES (?,?) ", (city.lower(), price,))
        conn.commit()


In [6]:
city_prices = [{"city": "london", "price": 199}, {"city": "lagos", "price": 600}]

for record in city_prices:
    insert_db_records_tool(city = record["city"], price = record["price"])

In [7]:
get_price_tool("lagos")

'The ticket price for lagos is 600.0'

In [13]:
GET_PRICE_TOOL = {
    "type": "function",
    "function": {
        "name": "get_price_tool",
        "description": "A tool to get the price of a ticket for a given city",
        "parameters": {
            "type": "object",
            "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city to get the ticket price for"
            }
        },
        "required": ["city"]
    }
    }
}

In [10]:
client = OpenAI()

In [29]:
history[2]

{'role': 'assistant',
 'content': ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_UYxG1LnJk3g5Y5CKzTndwpnj', function=Function(arguments='{"city":"London"}', name='get_price_tool'), type='function')])}

In [33]:
history = [
        {"role": "system", "content": "You help users look up city prices."},
        {"role": "user", "content": "What is the price for London?"}
    ]

response = client.chat.completions.create(

    model="gpt-4.1-mini",
     messages=history,
    tools=[GET_PRICE_TOOL],
    tool_choice="auto"
)

assistant_message = response.choices[0].message

history.append(assistant_message)

if response.choices[0].message.tool_calls:
    for tool_call in response.choices[0].message.tool_calls:
        if tool_call.function.name == "get_price_tool":
            tool_argument_object = json.loads(tool_call.function.arguments)["city"]
            tool_response = get_price_tool(tool_argument_object)
            history.append({"role": "tool","tool_call_id": tool_call.id, "content": tool_response})


final_response = client.chat.completions.create( 
    model="gpt-4.1-mini",
     messages=history,
)


        


In [42]:
def gradio_chat(message, history):

    message_history = [{"role": "system", "content": "You help users understand city prices. Never infer a price you must absolutley adhere to the tool given to you to search prices for cities."}]

    for msg in history:
        message_history.append(msg)

    message_history.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=message_history,
        tools = [GET_PRICE_TOOL],
        tool_choice = "auto"
    )

    assistant_message = response.choices[0].message
    message_history.append(assistant_message)  

    if response.choices[0].message.tool_calls:
        for tool_call in response.choices[0].message.tool_calls:
            if tool_call.function.name == "get_price_tool":
                tool_argument_object = json.loads(tool_call.function.arguments)["city"]
                tool_response = get_price_tool(tool_argument_object)
                message_history.append({"role": "tool","tool_call_id": tool_call.id, "content": tool_response})
    
        final_response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=message_history
        )
        return final_response.choices[0].message.content
    return assistant_message.content






In [43]:
gr.ChatInterface(fn=gradio_chat, title="City Price Bot", description="Ask me about ticket prices for different cities!").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [32]:
print(final_response.choices[0].message.content)

The price for London is 199.0.
